# Post process data extracted using LLMs
1. Load data
2. Convert to correct format (numeric for numerical columns)
3. Eventually explode lists for geocoding?
4. Geocoding
5. Sanity checks

In [20]:
import numpy as np
import pandas as pd
import geopandas as gpd
import geopy as gpy
import time
import itertools
import regex as re
from matplotlib import pyplot as plt
from src.data import *
from src.text_processing_functions import *
from src.plot_functions import *
from src.post_process_functions import *
from src.geocoding import *
from src.hazard_def import *
from src.impact_def import *
from src.sanity_checks import *



In [21]:
#load data (model)

res_savename = "labelled_reports_turnoff_subtype_val_meta-llama_llama-4-scout-17b-16e-instruct_v131025.csv"
#"labelled_reports_turnoff_subtype_val_llama-3.3-70b-versatile_v141025.csv"
#"labelled_reports_meta-llama_llama-4-scout-17b-16e-instruct_v230925.csv"#"labelled_reports_meta-llama_llama-4-scout-17b-16e-instruct_v230925.csv"
response_df = pd.read_csv(DATA_OUT_LLMS /res_savename)

#load data (labelled)
#res_savename = "labelled_reports_impacts_all_v080925.csv"
#response_df = pd.read_csv(DATA_LABELLED / res_savename)

savename = "test_reclass_subtype_post_processed_" + res_savename


In [22]:
#get rid of nans
response_df_proc = cp.deepcopy(response_df)

response_df_proc = response_df_proc.dropna(subset=["nathaz_text"]) if "nathaz_text" in response_df_proc.columns else response_df_proc

In [23]:
#process impactValue
response_df_proc = response_df_proc.apply(parse_impact_value_precision, axis=1)

/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:170: RuntimeWarning: All-NaN slice encountered
  min_value = np.nanmin(all_values)
/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:171: RuntimeWarning: All-NaN slice encountered
  max_value = np.nanmax(all_values)


In [24]:
#convert numerical columns
num_cols = ["impactValue", "impactValueMin", "impactValueMax","startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]
list_cols = ["country","location", "hazards", "valueAnnotation", "locationAnnotation", "dateAnnotation", "hazardsAnnotation", "annotation"]
list_cols = [key for key in list_cols if key in response_df_proc.columns]
response_df_proc = format_output(response_df_proc, num_cols=num_cols, list_cols=list_cols)



In [25]:
#add iso3
response_df_proc["country_iso3"] = response_df_proc["country"].apply(list_country_name_to_iso3)
response_df_proc["country_iso3_kw"] = response_df_proc["country_kw"].apply(list_country_name_to_iso3) if "country_kw" in response_df_proc.columns else None

In [26]:
#format units
from spacy.lang.en import English
from spacy.lang.punctuation import TOKENIZER_PREFIXES, TOKENIZER_SUFFIXES, TOKENIZER_INFIXES
from spacy.lang.en import TOKENIZER_EXCEPTIONS
from spacy.tokenizer import Tokenizer
from spacy.util import compile_prefix_regex, compile_suffix_regex, compile_infix_regex

## Post process
1. Reclassify hazards
2. Reclassify impactSubtypes
3. Reclassify units

In [27]:
#reclassify impacType
#response_df_proc["impactSubtype_orig"] = response_df_proc["impactSubtype"]
response_df_proc = response_df_proc.apply(reclassify_impact_subtype, axis=1)
response_df_proc = response_df_proc[response_df_proc["impactSubtype"] != "Unknown"]


In [28]:
response_df_proc["impactSubtype"].value_counts()

impactSubtype
Affected People                                    44
Other Human Impacts                                40
Residential Buildings                              37
Other Infrastructural Impacts                      26
Crop Production and Forestry                       23
Human Health and Wellbeing                         22
Access to Food                                     20
Access to Water, Sanitation, and Hygiene           20
Displaced People                                   19
Access to Healthcare                               18
Homeless People                                    16
Affected Livestock and Animals                     15
Water Quality and Availability                     15
Human Deaths                                       14
Agricultural Infrastructure                        13
Other Economic Activity & Livelihood Production    13
Road Infrastructure                                10
Other Service Access Impacts                        9
Water, Sanitat

In [29]:
#reclassify hazard
response_df_proc["hazards_orig"] = response_df_proc["hazards"]
response_df_proc = response_df_proc.apply(reclassify_hazard, hazard_kw_reclass=hazard_kw_reclass, axis=1)
response_df_proc.hazards.value_counts()

hazards
[Flood]                                                                                                                122
[Flood, Convective storm]                                                                                               46
[Drought]                                                                                                               41
[Flood, Mass movement]                                                                                                  26
[Flood, Tropical storm]                                                                                                 25
[Flood, Epidemic, Conflict]                                                                                             20
[Flood, Epidemic]                                                                                                       18
[Drought, Tropical storm]                                                                                               17
[Tropica

In [30]:
response_df_proc[response_df_proc["appealCode"] == "MDRSD034"][["hazards","hazards_orig", "flag_hazards_reclass"]]

,hazards,hazards_orig,flag_hazards_reclass
27,"[Flood, Epidemic]","[Flood, Epidemic]",False
28,"[Flood, Epidemic]","[Flood, Epidemic]",False
29,"[Flood, Epidemic, Conflict]","[Flood, Epidemic, Conflict]",False
30,"[Flood, Epidemic, Conflict]","[Flood, Epidemic, Conflict]",False
31,"[Flood, Epidemic, Conflict]","[Flood, Epidemic, Conflict]",False
32,"[Flood, Epidemic]","[Flood, Epidemic]",False
33,"[Flood, Epidemic, Conflict]","[Flood, Epidemic, Conflict]",False
34,"[Flood, Epidemic, Conflict]","[Flood, Epidemic, Conflict]",False
35,"[Flood, Epidemic, Conflict]","[Flood, Epidemic, Conflict]",False
36,[Flood],[Flood],False


In [31]:
response_df_proc

,impactSubtype,impactValue,impactValueMin,impactValueMax,impactValuePrecision,impactUnit,valueAnnotation,valid_errors_impactValue,country,location,...,country_kw,reportDate,reportLink,disasterType,nathaz_text,country_iso3,country_iso3_kw,flag_impactSubtype_reclass,hazards_orig,flag_hazards_reclass
0,Affected People,326788.0,NaN,NaN,exact,people,"[People Affected: 326,788 people]",0,[Pakistan],"[Balochistan, Khyber Pakhtunkhwa, Sindh]",...,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...,[PAK],PAK,False,[Flood],False
1,Injured People,584.0,NaN,NaN,exact,people,[The monsoon season caused 306 fatalities and ...,0,[Pakistan],"[Balochistan, Khyber Pakhtunkhwa, Sindh, Punja...",...,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...,[PAK],PAK,False,"[Flood, Convective storm]",False
2,Human Deaths,306.0,NaN,NaN,exact,people,[The monsoon season caused 306 fatalities and ...,0,[Pakistan],"[Balochistan, Khyber Pakhtunkhwa, Sindh, Punja...",...,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...,[PAK],PAK,False,"[Flood, Convective storm]",False
3,Displaced People,9500.0,NaN,NaN,exact,residents,"[Sindh experienced acute urban flooding, parti...",0,[Pakistan],"[Badin, Dadu, Jacobabad]",...,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...,[PAK],PAK,False,[Flood],False
4,Homeless People,15000.0,NaN,NaN,exact,houses,"[In response to this situation, the government...",0,[Pakistan],"[Sindh, Balochistan, Khyber Pakhtunkhwa]",...,Pakistan,2025-03-28,https://go-api.ifrc.org/api/downloadfile/88815...,Flood,['DREF Final Report Pakistan Flood August 2024...,[PAK],PAK,False,[Flood],False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,Other Human Impacts,1.0,NaN,NaN,exact,NaN,[The original plan was to hire an appeal coord...,1,[Guatemala],"[El Quiché, Western Temperate Highlands]",...,Guatemala,2016-11-14,https://adore.ifrc.org/Download.aspx?FileId=15...,Drought,"['P a g e | 1 Operation Update no.', '2 Operat...",[GTM],GTM,False,[Drought],False
484,Other Human Impacts,130.0,NaN,NaN,exact,training sessions,[From 130 to 50 training sessions in hygiene p...,0,[Guatemala],"[El Quiché, Western Temperate Highlands, Guate...",...,Guatemala,2016-11-14,https://adore.ifrc.org/Download.aspx?FileId=15...,Drought,"['P a g e | 1 Operation Update no.', '2 Operat...",[GTM],GTM,False,[Drought],False
485,Other Human Impacts,13.0,NaN,NaN,exact,%,"[Appeal Coverage to date: 13 % (269,543 CHF)]",0,[Guatemala],"[El Quiché, Western Temperate Highlands]",...,Guatemala,2016-11-14,https://adore.ifrc.org/Download.aspx?FileId=15...,Drought,"['P a g e | 1 Operation Update no.', '2 Operat...",[GTM],GTM,False,[Drought],False
486,Crop Production and Forestry,NaN,NaN,NaN,approx,NaN,[According to the Food Security Outlook Update...,2,[Guatemala],"[Western Temperate Highlands, low-lying areas ...",...,Guatemala,2016-11-14,https://adore.ifrc.org/Download.aspx?FileId=15...,Drought,"['P a g e | 1 Operation Update no.', '2 Operat...",[GTM],GTM,False,[Drought],False


In [32]:
wrong_haz = response_df_proc.explode("hazards", ignore_index=True).copy()
wrong_haz = wrong_haz[wrong_haz["hazards"] == "Unknown"]
wrong_haz["hazards"]

Series([], Name: hazards, dtype: object)

### Units reclassification
1. Standardize units i.e. metric units to SI or common units.
2. Determine unit typology (e.g. distance, surface, weight, percent) and special units (money, deaths)
3. Convert non-metric units e.g. households to people, USD to CHF
4. Standardize non-metric units i.e. person, children to people, hospitals to health facilities

TO DOS
1. Handle deaths so that they dont go in other categories
2. Handle targeted people
3. ~~Handle currencies~~
4. When unknown unit, try to infer it using kw for other impact subtype and reclass to other subtype if match
5. Handle damage vs destroyed houses
6. Parse correctly impact values min and maxs

In [33]:
#response_df_proc["impactValue"] = response_df_proc["impactValueOrig"]
#response_df_proc["impactUnit"] = response_df_proc["impactUnitOrig"]

In [34]:
def infer_unit_from_annotation(x):
    annotation = x["annotation"] if x["annotation"] else x["valueAnnotation"]
    value = x["impactValue"]
    unit = x["impactUnit"]
    #if there is no value we return what's orignally there
    if pd.isnull(value):
        return pd.Series({"impactValue": value, "impactUnit": unit})

    #format numbers in annotation
    annotation = replace_commas_in_numbers(annotation)
    annotation = replace_count_suffixes(annotation)
    annotation = replace_numbers(annotation)

    #format value
    value = format_number(value)

    #find value in annotation
    kw_value = re.search(r"\{value\}", annotation, re.IGNORECASE)
    unit_kw = [kw for kw in unit_kw_reclass.keys() if re.search(unit_kw_reclass[kw], annotation, re.IGNORECASE)]
    if len(unit_kw) == 1:
        return pd.Series({"impactValue": value, "impactUnit": unit_kw[0]})
    else:
        return "Unknown"


In [35]:
force_unit_to_subtype = False #whether or not we want to force unit to default unit of subtype when unknown unit
reclass_subtype = True
#keep orig unit and value for comparison
response_df_proc["impactValueOrig"] = response_df_proc["impactValue"]
response_df_proc["impactUnitOrig"] = response_df_proc["impactUnit"]
## Units reclassification
#replace numbers in units
response_df_proc = response_df_proc.apply(replace_numbers_unit, axis=1)
#convert money
response_df_proc = response_df_proc.apply(convert_monetary_units, axis=1)
#standardize metric units
response_df_proc = response_df_proc.apply(standardize_metric_units, axis=1)
#assign unit type (e.g. surface, volume, mass)
response_df_proc = response_df_proc.apply(assign_unit_type, axis=1)
#harmonize non metric units
response_df_proc = response_df_proc.apply(harmonize_units, axis=1)
#convert convertible (non-money) units
response_df_proc = response_df_proc.apply(convert_unit, axis=1)
#reclassify units
response_df_proc = response_df_proc.apply(reclassify_units, force_unit_to_subtype=force_unit_to_subtype, reclass_subtype=reclass_subtype, axis=1)
#normalize people units
response_df_proc = response_df_proc.apply(normalize_people_unit, axis=1)


/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:536: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  if len(pd.unique(units_parsed)) > 1:#only do assignment if all units are the same
/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:539: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  elif len(pd.unique(units_parsed)) == 1:
/Users/lseverino/Documents/PhD/Projects/Como/como_project4/src/post_process_functions.py:536: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  if len(pd.unique(units_parsed)) > 1:#only do assignment if all units are the same
/Users/lseverino/Documents/PhD/Projects/Como/como_project4/sr

Reclassified subtype from Homeless People to Residential Buildings with unit reclass homes and orig unit homes
Reclassified subtype from Agricultural Infrastructure to Affected Livestock and Animals with unit reclass affected animals and orig unit livestock
Reclassified subtype from Road Infrastructure to Displaced People with unit reclass displaced and orig unit people displaced
Reclassified subtype from Water Quality and Availability to Infected and Ill People with unit reclass cases and orig unit cases
Reclassified subtype from Other Infrastructural Impacts to Water, Sanitation, and Hygiene Infrastructure with unit reclass WASH structures and orig unit latrines
Reclassified subtype from Water Quality and Availability to Water, Sanitation, and Hygiene Infrastructure with unit reclass WASH structures and orig unit water treatment plants
Reclassified subtype from Other Infrastructural Impacts to Road Infrastructure with unit reclass roads and orig unit roads
Reclassified subtype from O

In [19]:
force_unit_to_subtype = False #whether or not we want to force unit to default unit of subtype when unknown unit
reclass_subtype = True
test_df = pd.DataFrame({"impactUnit": "nomads families", "unit_type": "%", "impactValue": 100, "impactValueMin": np.nan, "impactValueMax":200}, index=[0])
#harmonize non metric units
test_df = test_df.apply(harmonize_units, axis=1)
#convert convertible (non-money) units
test_df = test_df.apply(convert_unit, unit_converter=unit_converter, axis=1)
#reclassify units
test_df = test_df.apply(reclassify_units, unit_kw_reclass=unit_kw_reclass, default_subtype_unit=default_subtype_unit, force_unit_to_subtype=force_unit_to_subtype, reclass_subtype=reclass_subtype, axis=1)
#normalize people units
test_df = test_df.apply(normalize_people_unit, axis=1)
test_df

,impactUnit,unit_type,impactValue,impactValueMin,impactValueMax,flag_unit_harmonization,flag_unit_conversion,flag_unit_nonstd,flag_reclass_subtype_from_unit
0,nomads people,%,300.0,NaN,600.0,False,True,True,False


In [20]:
test_df

,impactUnit,unit_type,impactValue,impactValueMin,impactValueMax,flag_unit_harmonization,flag_unit_conversion,flag_unit_nonstd,flag_reclass_subtype_from_unit
0,nomads people,%,300.0,NaN,600.0,False,True,True,False


In [22]:
## Post conversion flags
country_pop = pd.read_csv(DATA_PATH / ("API_SP.POP.TOTL_DS2_en_csv_v2_131993/"+"API_SP.POP.TOTL_DS2_en_csv_v2_131993.csv"),sep=',', header=2).dropna(how="all",axis=1)
# country_pop = pd.read_csv(os.path.join(DATA_PATH, "API_SP.POP.TOTL_DS2_en_csv_v2_131993", "API_SP.POP.TOTL_DS2_en_csv_v2_131993.csv"),sep=',', header=2).dropna(how="all",axis=1)
response_df_proc["flag_pop_cntry"] = response_df_proc.apply(pop_cntry_check, country_pop=country_pop, axis=1)
response_df_proc["flag_value_no_unit"] = response_df_proc.apply(flag_value_no_unit, axis=1)
response_df_proc["flag_partial_unit"] = response_df_proc.apply(flag_partial_unit, axis=1)
response_df_proc["flag_percent"] = response_df_proc.apply(flag_percent, axis=1)

 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023
 Defaulting to year 2023


In [24]:
response_df_proc[response_df_proc["flag_unit_nonstd"]]

,impactSubtype,impactValue,impactValueMin,impactValueMax,impactValuePrecision,impactUnit,valueAnnotation,valid_errors_impactValue,country,location,...,unit_type,flag_unit_type,flag_unit_harmonization,flag_unit_conversion,flag_unit_nonstd,flag_reclass_subtype_from_unit,flag_pop_cntry,flag_value_no_unit,flag_partial_unit,flag_percent
50,Healthcare Infrastructure,119.000000,NaN,NaN,exact,incidents of heavy rain,"[In total, 119 incidents of heavy rain were re...",0,[Sudan],"[Kassala, Gedaref, River Nile, Gazira, Blue Ni...",...,other,False,False,False,True,False,NaN,False,False,False
98,Crop Production and Forestry,40.000000,NaN,NaN,exact,%,[Prices were 40% higher or more compared to la...,0,[Mozambique],[central and northern zones],...,%,False,False,False,True,False,NaN,False,True,False
101,Access to Food,3.000000,NaN,NaN,exact,%,[Mozambique National Institute of Statistics' ...,0,[Mozambique],"[Tete, Gaza, Manica, Inhambane]",...,%,False,False,False,True,False,NaN,False,True,False
118,Affected Livestock and Animals,300.000000,NaN,NaN,exact,people from nomads,[100 families from Nomads have been affected],0,[Algeria],[Tamanrasset],...,other,False,False,True,True,False,NaN,False,False,False
120,Access to Food,1000.000000,NaN,NaN,exact,parcels,"[despite the distribution of 1,000 parcels]",0,[Algeria],"[Béchar, Elbayadh, Beni Abbes, Tamanrasset, Ti...",...,other,False,False,False,True,False,NaN,False,False,False
156,Agricultural Infrastructure,37.810000,NaN,NaN,exact,km**2,"[3,781 hectares of crops damaged]",0,[Benin],"[Couffo, Adoukandji, Ahodjinnako, Ahomadegbe, ...",...,km**2,False,False,False,True,False,NaN,False,True,False
157,Crop Production and Forestry,37.810000,NaN,NaN,exact,km**2,"[3,781 hectares of crops damaged]",0,[Benin],"[Couffo, Adoukandji, Ahodjinnako, Ahomadegbe, ...",...,km**2,False,False,False,True,False,NaN,False,True,False
206,Access to Education,1600.000000,NaN,NaN,exact,children,"[school kits, including exercise books, pens a...",0,[Rwanda],[],...,other,False,False,False,True,False,NaN,False,False,False
309,Crop Production and Forestry,112.167168,NaN,NaN,exact,km**2,[October November December short rains season ...,0,[Kenya],[],...,km**2,False,False,False,True,False,NaN,False,True,False
322,Other Human Impacts,76.600000,NaN,NaN,exact,%,[Reliance on rainfed agriculture drives the hi...,0,[Zambia],[Rural Zambia],...,%,False,False,False,True,False,NaN,False,True,False


In [35]:
response_df[response_df["appealCode"]=="MDRUG050"]

,impactSubtype,impactValue,impactValueMin,impactValueMax,impactValuePrecision,impactUnit,valueAnnotation,country,location,locationAnnotation,...,endDay,dateAnnotation,hazards,hazardsAnnotation,appealCode,country_kw,reportDate,reportLink,disasterType,nathaz_text
156,Affected People,69.0,NaN,NaN,exact,NaN,"['In total, by August2024, the floods have aff...",['Uganda'],"['Central Region', 'Eastern Region', 'Western ...","['In total, by August2024, the floods have aff...",...,NaN,"['In total, by August2024, the floods have aff...","['Flood', 'Convective storm', 'Mass movement']","['In April2024, the Eastern Uganda-Elgon regio...",MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['DREF Operational Update Uganda_Floods Some o...
157,Affected People,18.0,NaN,NaN,exact,NaN,"['18,323 people were affected, including thous...",['Uganda'],"['Mbale', 'Kapchorwa', 'Bulambuli', 'Bukedea',...","['In April2024, the Eastern Uganda-Elgon regio...",...,NaN,"['In April2024, the Eastern Uganda-Elgon regio...","['Flood', 'Convective storm', 'Mass movement']","['In April2024, the Eastern Uganda-Elgon regio...",MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['DREF Operational Update Uganda_Floods Some o...
158,Affected People,11.0,NaN,NaN,exact,NaN,"['11,775 individuals (2,355Households) have be...",['Uganda'],['Ntoroko district'],['The recent floods in Ntoroko District that h...,...,NaN,['The recent floods in Ntoroko District that h...,['Flood'],['The recent floods in Ntoroko District that h...,MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['DREF Operational Update Uganda_Floods Some o...
159,Injured People,166.0,NaN,NaN,exact,NaN,['This flooding also significantly affected he...,['Uganda'],"['Butaleja district, Bukedi Sub County']",['This flooding also significantly affected he...,...,NaN,"[""On April3rd, floods hit Bulambuli's Bunambut...","['Flood', 'Convective storm']","[""On April3rd, floods hit Bulambuli's Bunambut...",MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['DREF Operational Update Uganda_Floods Some o...
160,Injured People,9.0,NaN,NaN,exact,NaN,['This left109 people homeless and worsened sa...,['Uganda'],"['Mbale District', 'Bukasakya', 'Bungokho']",['This left109 people homeless and worsened sa...,...,NaN,['This left109 people homeless and worsened sa...,['Convective storm'],"['In Mbale District, Bukasakya, and Bungokho, ...",MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['DREF Operational Update Uganda_Floods Some o...
161,Injured People,6.0,NaN,NaN,exact,NaN,"[""On April3rd, floods hit Bulambuli's Bunambut...",['Uganda'],"[""Bulambuli's Bunambutye sub-county""]","[""On April3rd, floods hit Bulambuli's Bunambut...",...,NaN,"[""On April3rd, floods hit Bulambuli's Bunambut...",['Flood'],"[""On April3rd, floods hit Bulambuli's Bunambut...",MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['DREF Operational Update Uganda_Floods Some o...
162,Displaced People,10.0,NaN,NaN,exact,NaN,"['In total, by August2024, the floods have aff...",['Uganda'],"['Central Region', 'Eastern Region', 'Western ...","['In total, by August2024, the floods have aff...",...,21.0,"['In April2024, the Eastern Uganda-Elgon regio...","['Flood', 'Convective storm', 'Mass movement']","['In April2024, the Eastern Uganda-Elgon regio...",MDRUG050,Uganda,2024-08-30 00:00:00,https://adore.ifrc.org/Download.aspx?FileId=83...,Flood,['DREF Operational Update Uganda_Floods Some o...
163,Displaced People,3.0,NaN,NaN,exact,NaN,"['3,080 Families were displaced and rendered h...",['Uganda'],"['Bweramule Sub-County', 'Butungama Sub-County...","['In Bweramule Sub-County,714 households were ...",...,21.0,"['In April2024, the Eastern Uganda-Elgon regio...","['Flood', 'Convective storm', 'Mass movement']","['In April2024, t

In [34]:
response_df_proc[response_df_proc["appealCode"]=="MDRUG050"][["impactSubtype","impactValueOrig", "impactValue", "impactUnitOrig", "impactUnit","valueAnnotation"]]

,impactSubtype,impactValueOrig,impactValue,impactUnitOrig,impactUnit,valueAnnotation
156,Affected People,69.0,69.0,NaN,nan,"[In total, by August2024, the floods have affe..."
157,Affected People,18.0,18.0,NaN,nan,"[18,323 people were affected, including thousa..."
158,Affected People,11.0,11.0,NaN,nan,"[11,775 individuals (2,355Households) have bee..."
159,Injured People,166.0,166.0,NaN,nan,[This flooding also significantly affected hea...
160,Injured People,9.0,9.0,NaN,nan,[This left109 people homeless and worsened san...
161,Injured People,6.0,6.0,NaN,nan,"[On April3rd, floods hit Bulambuli's Bunambuty..."
162,Displaced People,10.0,10.0,NaN,nan,"[In total, by August2024, the floods have affe..."
163,Displaced People,3.0,3.0,NaN,nan,"[3,080 Families were displaced and rendered ho..."
164,Homeless People,1.0,1.0,NaN,nan,[This event also damaged six water facilities ...
165,Homeless People,327.0,327.0,NaN,nan,"[In Bweramule Sub-County,714 households were a..."


In [17]:
#test replace numbers


In [18]:
from price_parser import Price
test_price_true1 = "1,500 USD"
parsed_price1 = Price.fromstring(test_price_true1)
print(parsed_price1)

test_price_true2 = "735 billion CHF"
parsed_price2 = Price.fromstring(test_price_true2)
print(parsed_price2)

test_price_wrong = "1,500"
parsed_price2 = Price.fromstring(test_price_wrong)
print(parsed_price2)

test_price_wrong = "1,500 PHP"
parsed_price2 = Price.fromstring(test_price_wrong)
print(parsed_price2)

Price(amount=Decimal('1500'), currency='USD')
Price(amount=Decimal('735'), currency='CHF')
Price(amount=Decimal('1500'), currency=None)
Price(amount=Decimal('1500'), currency='PHP')


In [19]:
from currency_converter import CurrencyConverter
c = CurrencyConverter()
test_conv_pass =(1500, "USD")
DEF_CUR = "EUR"
conv_price_pass = c.convert(test_conv_pass[0], test_conv_pass[1], DEF_CUR)
print(conv_price_pass)
test_conv_fail = (1500, "foo")
try:
    conv_price_fail = c.convert(test_conv_fail[0], test_conv_fail[1], DEF_CUR)
    print(conv_price_fail)
except Exception as e:
    print(e)
test_conv_fail2 = ("asda", "USD")
try:
    conv_price_fail2 = c.convert(test_conv_fail2[0], test_conv_fail2[1], DEF_CUR)
    print(conv_price_fail2)
except Exception as e:
    print(e)

1287.1117212974086
foo is not a supported currency
could not convert string to float: 'asda'


In [20]:
parsed_price1.currency

'USD'

In [21]:
unit_type = "kg"
unit = "kg of crops"
[unit_corr for unit_corr in unit_kw_reclass.keys() if re.search(unit_kw_reclass[unit_corr], unit, re.IGNORECASE)]


['crop production and forestry']

In [22]:
test_df = pd.DataFrame({
        "reportDate": ["2020-01-01", "2020-01-01", "2020-01-01"],
        "impactSubtype": ["Affected People", "Crop Production and Forestry", "Crop Production and Forestry"],
        "impactValue": [1000,1000,100],
        "impactUnit": ["families", "kg of crops","hectares of crops"]})
#replace numbers in units
test_df[["impactValue", "impactUnit"]] = test_df.apply(replace_numbers_unit, axis=1)
#convert money
test_df[["impactValue", "impactUnit"]] = test_df.apply(convert_monetary_units, axis=1)
#standardize SI units
test_df[["impactValue", "impactUnit"]]  = test_df.apply(standardize_metric_units, std_unit_kw_reclass=std_unit_kw_reclass, unit_mapping=unit_mapping, axis=1)
test_df = test_df.apply(convert_unit, unit_converter=unit_converter, axis=1)
test_df["unit_type"] = test_df.apply(assign_unit_type, unit_type_kw_reclass=unit_type_kw_reclass, axis=1)
test_df["impactUnit"] = test_df.apply(reclassify_units, unit_kw_reclass=unit_kw_reclass, default_subtype_unit=default_subtype_unit, force_unit_to_subtype=force_unit_to_subtype, axis=1)
test_df

,reportDate,impactSubtype,impactValue,impactUnit,unit_type
0,2020-01-01,Affected People,3000.0,people,other
1,2020-01-01,Crop Production and Forestry,1000.0,kg of crop production and forestry,kg
2,2020-01-01,Crop Production and Forestry,1.0,km**2 of crop production and forestry,km**2


In [23]:
# save
response_df_proc.to_csv(DATA_OUT_PROC / savename, index=False)

In [24]:
response_df_proc[["impactValue", "impactUnit", "impactValueOrig", "impactUnitOrig", "valueAnnotation"]]

,impactValue,impactUnit,impactValueOrig,impactUnitOrig,valueAnnotation
0,3.267880e+05,people,326788.0,people,"[People Affected:326,788 people]"
1,5.840000e+02,people,584.0,people,[The monsoon season caused306 fatalities and58...
2,3.060000e+02,people,306.0,people,[The monsoon season caused306 fatalities and58...
3,9.500000e+03,people,9500.0,residents,"[Sindh experienced acute urban flooding, parti..."
4,1.500000e+04,homes,15000.0,houses,"[In regions such as KP and Sindh, particularly..."
...,...,...,...,...,...
311,1.000000e+00,people,1.0,person,[NADMA reported that the state of Johor suffer...
312,3.294680e+05,people,329468.0,people,[According to the Department of Social Welfare...
313,1.277700e+04,homes,12777.0,houses,"[According to government data,12,777 houses we..."
314,6.791483e+07,eur,74.0,CHF million,"[Furthermore, damage to agriculture sector amo..."


In [25]:
response_df_proc.groupby("appealCode").count()

,impactSubtype,impactValue,impactValueMin,impactValueMax,impactValuePrecision,impactUnit,valueAnnotation,country,location,locationAnnotation,...,reportLink,disasterType,nathaz_text,country_iso3,country_iso3_kw,impactSubtype_orig,hazards_orig,impactValueOrig,impactUnitOrig,unit_type
appealCode,,,,,,,,,,,,,,,,,,,,,
MDRBD022,9,7,0,0,7,9,9,9,9,9,...,9,9,9,9,9,9,9,7,7,9
MDRBJ019,10,5,0,0,5,10,10,10,10,10,...,10,10,10,10,10,10,10,5,5,10
MDRCM039,15,13,0,0,13,15,15,15,15,15,...,15,15,15,15,15,15,15,13,13,15
MDRCN006,13,13,0,0,13,13,13,13,13,13,...,13,13,13,13,13,13,13,13,13,13
MDRDZ011,13,6,0,0,6,13,13,13,13,13,...,13,13,13,13,13,13,13,6,6,13
MDRGE019,15,3,0,0,3,15,15,15,15,15,...,15,15,15,15,15,15,15,3,3,15
MDRIQ014,8,5,0,0,5,8,8,8,8,8,...,8,8,8,8,8,8,8,5,5,8
MDRKE058,31,27,0,0,27,31,31,31,31,31,...,31,31,31,31,31,31,31,27,27,31
MDRMY003,3,3,0,0,3,3,3,3,3,3,...,3,3,3,3,3,3,3,3,3,3
